These functions currently reside in the "navis_functions" branch of the "ac_segmentation" repo.

In [1]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
import navis
from joblib import dump, load
import os
import neuroglancer
import requests
import glob
from cloudvolume import CloudVolume, Skeleton
import numpy as np
from pathlib import Path

from ac_segmentation.segmentation import predict_array, predict_zarr_strips
from ac_segmentation.postprocess import postprocess_kimi_array, postprocess_kimi_zarr_strips
from ac_segmentation.reconnect_stack_navis import reconnect, swc_multi_to_single_subdir, load_swc, save_swc
from ac_segmentation.preprocess import lut_preprocess_array
from ac_segmentation.neurotorch.datasets.datatypes import (BoundingBox, Vector)

/home/connor.laughland/ENTER/envs/seg3/lib/python3.8/site-packages/ac_segmentation/neurotorch/nets/layers.py:40: UserWarning: nn.init.kaiming_normal is now deprecated in favor of nn.init.kaiming_normal_.
  init.kaiming_normal(self.conv.weight)
/home/connor.laughland/ENTER/envs/seg3/lib/python3.8/site-packages/ac_segmentation/neurotorch/nets/layers.py:42: UserWarning: nn.init.constant is now deprecated in favor of nn.init.constant_.
  init.constant(self.conv.bias, 0)
/home/connor.laughland/ENTER/envs/seg3/lib/python3.8/site-packages/ac_segmentation/neurotorch/nets/layers.py:81: UserWarning: nn.init.kaiming_normal is now deprecated in favor of nn.init.kaiming_normal_.
  init.kaiming_normal(self.conv.weight)


In [2]:
def predict_zarr_strips(zarr_dir, outdir, strip_range, checkpt_file, z_crop = None,  level = 0, max_pix = 30000, 
                           iter_size = [32, 32, 32], stride = [64, 64, 64]):

    for strip in range(strip_range[0], strip_range[1]+1):
        pos_dir = outdir + 'Pos' + str(strip) + "/"
        os.makedirs(pos_dir, exist_ok=True)

        #load zarr
        f_path = zarr_dir + 'highres_Pos' + str(strip)
        data = zarr.load(f_path)
        data = data[level]

        #reformat, index, and adjust pixel intensity
        data = np.transpose(data[0,0,:,:,:])
        if z_crop != None:
            data = data[:,:,z_crop[0]:z_crop[1]]
        data = lut_preprocess_array(data, max_pix)

        #create empty array for output
        zarr_arr = np.zeros((data.shape[0],data.shape[1],data.shape[2]))

        #run segmentation
        out_arr = predict_array(checkpt_file, './', data, output_type = 'volume', 
                                    iter_size= BoundingBox(Vector(0, 0, 0), Vector(iter_size[0], iter_size[1], iter_size[2])), 
                                    stride = Vector(stride[0], stride[1], stride[2]))
        
        #save array to zarr
        zarr.save(pos_dir + 'Pos' + str(strip) + '_Segmented.zarr', out_arr)
        print("Position " + str(strip) + " Complete")

In [19]:
def postprocess_kimi_zarr_strips(in_dir, outdir, sc, cl, strip_range, bound_box,
                            prob_thresh = 0.1, match_query_dis = 20, min_collin=0.1, size_thresh = 500, thresh = 0.05):
    
    for strip in range(strip_range[0], strip_range[1]+1):
        pos_dir = in_dir + 'Pos' + str(strip) + "/"
        seg_data = zarr.open(pos_dir + 'Pos' + str(strip) + '_Segmented.zarr')
        test_arr = np.transpose(seg_data)
        
        #run skeletonization
        postprocess_kimi_array(outdir = pos_dir, stack = test_arr, bound_box = [bound_box[0], bound_box[1], bound_box[2]], chunk_size = [512, 512, 64], overlap = [512, 512, 64], threshold=thresh, size_threshold=size_thresh, check_rad=True)
        skel_dir = pos_dir + "/swc_files_KIMI/"
        skels = os.listdir(skel_dir)

        #Convert all SWCs to a single SWC
        all_skel = navis.read_swc(pos_dir, include_subdirs=True)
        swc_multi_to_single_subdir(pos_dir, pos_dir + 'consolidated.swc' )
        
        #Break and reconnect skeletons
        os.makedirs(pos_dir + "Reconnected/", exist_ok=True)
        skels_rec = reconnect(infile = pos_dir + 'consolidated.swc', \
                                    swc_outdir = pos_dir + "Reconnected/", cl = cl, sc = sc, \
                                    min_nodes = 10, prob_thresh = prob_thresh, query_dis = match_query_dis, min_collin=min_collin)
        
        #Convert all SWCs to a single SWC
        os.makedirs(outdir + "Skeletons/", exist_ok=True)
        swc_multi_to_single_subdir(pos_dir + "Reconnected/reconnected_skeletons/",\
                                   outdir + "Skeletons/Pos" + str(strip) + "_Skels.swc" ) 
        
        print("Position " + str(strip) + " Complete!")

In [26]:
###Run Segmentation
zarr_dir = "/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S33_230413_highres/H17_x55_S33_230413_highres.zarr/"
indir = '/ACdata/Users/connorl/Test/'
outdir = '/ACdata/Users/connorl/Test/'
strip_range = [53,53]
checkpt_file = "/home/russelt/some_ckpt.ckpt"

predict_zarr_strips(zarr_dir, indir, strip_range, checkpt_file, z_crop = [10000,16000], level = 1)

Position 53 Complete


In [27]:
###Run Skeletonization
sc = load("/ACdata/Users/connorl/Models/scaler.joblib")
cl = load("/ACdata/Users/connorl/Models/LR_1.joblib")

postprocess_kimi_zarr_strips(in_dir = indir, outdir = outdir, sc = sc, cl = cl, strip_range = [53,53], bound_box = [30000, 30000, 30000], size_thresh = 500)

/home/connor.laughland/ENTER/envs/seg3/lib/python3.8/site-packages/sklearn/base.py:347: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 0.23.1 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/connor.laughland/ENTER/envs/seg3/lib/python3.8/site-packages/sklearn/base.py:347: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 0.23.1 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Skeletonizing Labels: 100%|███████████████████████████████████████████████████| 1358/1358 [00:02<00:00, 555.02it/s]


Importing:   0%|          | 0/1358 [00:00<?, ?it/s]

Smoothing:   0%|          | 0/1871 [00:00<?, ?it/s]

Pairs Merged:  14


Writing:   0%|          | 0/1449 [00:00<?, ?it/s]

Position 53 Complete!
